# Lantern LAGN First Cut Filter

*Based on notebook written by Zach Gillis, available at https://github.com/drphilmarshall/lantern/tree/main/notebooks*

*This revised notebook is written by Natascha Barac*

*This notebook is adapted to the cosmicAI workshop use by Padma Venkatraman*

*Date last updated: 28 May 2026*

*Collaborators on this work: Phil Marhsall, Shenming Fu, Padma Venkatraman, Paras Sharma, and many more!*

This notebook prepares our *First Cut* filter, which selects lensed-AGN (LAGN) from the Rubin Observatory alert stream -- which you can look at here: https://antares.noirlab.edu/loci -- using difference image analysis sources (DIASources). The trained filter is implemented at ANTARES, and sources which pass the filter there are tagged as lantern targets (https://antares.noirlab.edu/tags).



## Overview

Rubin's difference imaging pipeline subtracts a template image from each visit image, producing a "difference image" where only flux changes appear. Sources detected in these difference images are called DIASources. The full DP1 dataset contains 1.4 million DIASources within our three target survey fields. Our goal is to progressively filter this sample down to sources consistent with spatially extended flux differences that could be lensed AGN.

Lensed AGN are predicted to appear and be efficiently detected as spatially extended sources in difference images from optical imaging surveys (Kochanek et al. 2006). This motivates our use of extendedness metrics as the primary discriminating features in the simple cuts filter.

## Dataset

Our dataset (`combined_training_data_v2.0.7`) is built using the DIASources of simulated LAGN injected in the ECDFS field, which are assigned ground-truth `lagn` labels. Each LAGN injected is assigned a unique `lens_id` value so that we can identify unique lens systems.

Our non-LAGN DIASources are the full set of DP1 DIASources in the ECDFS field. Note that we assume all of these sources are non-LAGN.

| Field | RA | Dec | Notes |
|---|---|---|---|
| ECDFS | 53.16° | −28.10° | Extended Chandra Deep Field South |

Non-LAGN have been clustered into sky positions of 3arcsec. Each sky position has been assigned a unique `lens_id` as well, so that we can stratify the training and test sets based on unique sources. Non-LAGN can be distinguished from LAGN because they are assigne negative `lens_id` values.

## Model

This notebook trains an XGBoost binary classifier to distinguish LAGN DIASources from non-LAGN DIASources, valuates it on a held-out test set, and prepares a completeness/purity curve using estimated LSST survey statistics.

---

## Current Status

You can go to https://antares.noirlab.edu/loci and click on `latern_xgboost_t2.0.7_c0.95` under "Tags" on the left of the webpage. You will see that there are 113.3k tagged alerts! We expect to find ~100s of LAGN so far based on how much of the sky has been surveyed. You can see if you can make the classify better!

## 0. Make sure all the required packages are installed

In [ ]:
# import os
# import sys

# # Safely install requirements to the current notebook environment
# os.system(f"{sys.executable} -m pip install -r ../requirements.txt")


## 1. Imports & Data Loading

### Import Libraries

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import yaml
!pip install corner

from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, cross_val_score, StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    roc_curve, auc, confusion_matrix,
    precision_recall_curve, average_precision_score, matthews_corrcoef,
)

from sklearn.calibration import calibration_curve

### Load & Split Data

Loads `combined_training_data_{version}.csv`.

Each row per DIASource with `label=0` (non-LAGN DIASource, DP1) or `label=1` (LAGN DIASource, injected).

But be careful! Each row *is not a unique lens*! You want to make sure that your random forest has this information. When you use `StratifiedGroupKFold`, this allows you to pass in *groups*, and you want to make sure that each unique lens is passed in as a group.

In [11]:
from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/My Drive/Cosmic AI Summer School'
training_data = 'combined_training_data_v2.0.6_all_cols.csv'
df = pd.read_csv(training_data)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: 'combined_training_data_v2.0.6_all_cols.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/Cosmic AI Summer School'
training_data = 'combined_training_data_v2.0.6_all_cols.csv'
df = pd.read_csv(f'{path}/{training_data}')

### Exploratory data analysis:
<div style="color:green">
What does each column describe? There are probably a lot of unknown terminology in this work. This will be common with most datasets you explore. This notebook: https://dp1.lsst.io/tutorials/notebook/201/notebook-201-5.html might serve to explain some of the columns but also feel free to look things up! "Rubin" "DIASource" along with the column name should reveal some answers.

What are the correlations between different properties? You can try using seaborn pairplot to explore the different dimensions.
</div>

**Active features:** `band`, `psf_fwhm`, `snr`, `template_flux`, `scienceFlux`, `psfFlux`, `apFlux`, `temp_sci_flux_ratio`, `moment_ext`, `ellip_ext`, `flux_ext`, `extendedness`, `psfChi2`, `dipoleFitAttempted`, `dipoleChi2`, `dipoleLength`, `x_y_err`

A subset of features is selected from the candidate features in `combined_training_data_{version}.csv`. Currently, several features are commented out to reduce overfitting risk (`centroid_flag`, `trailLength`, `trailFlux`, and the dipole error/flux columns).


In [ ]:

FEATURES = [
    # Other
    'band',
    # 'centroid_flag',
    'psf_fwhm',
    'snr',

    # Flux
    'template_flux',
    'scienceFlux',
    'psfFlux',
    'apFlux',
    'temp_sci_flux_ratio',

    # Extended
    'moment_ext',
    'ellip_ext',
    'flux_ext',
    'extendedness',
    'psfChi2',
    # 'trailFlux',
    # 'trailLength',

    # Dipole
    # 'isDipole',
    'dipoleFitAttempted',
    'dipoleChi2',
    # 'dipoleFluxDiffErr',
    # 'dipoleMeanFlux',
    # 'dipoleMeanFluxErr',
    'dipoleLength',

    # Centroid
    'x_y_err',
]

<div style="color:green">

Look at all the columns! What are the datatypes? What columns could be helpful in identifying lensed AGN? Here is context for what lensed AGN look like: https://docs.google.com/presentation/d/1cweC0QvG8tYl1bo9FECRe6Q0PnF4WnU5uSscy58Xrkc/edit?slide=id.g3ed1d43efb5_0_8#slide=id.g3ed1d43efb5_0_8!

</div>

#### Band Weighting

The injected LAGN (label=1) and DP1 background (label=0) have different band distributions. Without correction, the model could implicitly learn band membership as a proxy for class label rather than the underlying physical features.

To address this, each training sample is assigned a weight inversely proportional to the count of its `(label, band)` cell, normalized so the mean weight is 1.

This ensures every `(label, band)` combination contributes equally to the loss. Because LAGN cells are ~400–800× smaller than non-LAGN cells, these weights also subsume the overall class imbalance correction. Weights are passed to `model.fit()` via `sample_weight` and recomputed per fold in the Section 5 cross-validation.

In [ ]:
df['band'] = pd.Categorical(df['band'])
band_categories = df['band'].cat.categories
band_categories
### what columns should you drop here? Remember leakage!
### You can chain commands in pandas: drop unwanted features and the select the desired features
X = df.drop(columns=[...])[...]

### which column has the "true" information?
y = ...
### how to assign different groups? why do we do this?
groups = ...

We use `StratifiedGroupKFold` to prevent data leakage (keep unique lenses in the same subset), and preserve the class distribution in each fold. Why does `StratifiedGroupKFold` help achieve this?

---

The dataset is split 80% train / 20% test with `stratify=y` to preserve the class ratio in both splits.

---

XGBoost natively supports NaN values (present in dipole-specific columns in DIASources without attempted dipole fits) and categorical variables (i.e. `band`).


In [ ]:


# Split by group using StratifiedGroupKFold: 80% train_full, 20% test
sgkf = ...
### assign train and test indices
train_full_idx, test_idx = ...

# what is this second split for? for validation!
# Second split: from train_full, 80% train, 20% validation
X_train_full = X.iloc[train_full_idx]
y_train_full = y.iloc[train_full_idx]
groups_train_full = groups.iloc[train_full_idx]

sgkf2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=43)
train_idx_temp, val_idx_temp = next(sgkf2.split(X_train_full, y_train_full, groups=groups_train_full))

### every time you create a variable, you should LOOK AT IT!
# Map back to original indices
train_idx = train_full_idx[train_idx_temp]
val_idx = train_full_idx[val_idx_temp]

# Create final subsets
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

# Get groups for each split
groups_train = groups.iloc[train_idx]
groups_val = groups.iloc[val_idx]
groups_test = groups.iloc[test_idx]

print(f"\nSplits:")
print(f"  Train: {len(X_train):,} DIAsources, {groups_train[y_train==1].nunique():,} lenses, {groups_train[y_train==0].nunique():,} non-lenses")
print(f"  Val:  {len(X_val):,} DIAsources, {groups_val[y_val==1].nunique():,} lenses, {groups_val[y_val==0].nunique():,} non-lenses")
print(f"  Test:  {len(X_test):,} DIAsources, {groups_test[y_test==1].nunique():,} lenses, {groups_test[y_test==0].nunique():,} non-lenses")

In [ ]:
# Compute sample weights
def compute_band_weights(y_s, X_s):
    tmp = pd.DataFrame({'label': y_s.values, 'band': X_s['band'].values})
    cell_counts = tmp.groupby(['label', 'band'], observed=True)['label'].transform('count')
    w = 1.0 / cell_counts
    return (w / w.mean()).values

<div style="color:green">

What does the function `compute_band_weights` do? Why do we need to weight different bands differently? We answer this question earlier in the notebook, but it is important to understand why!

</div>

In [ ]:
sample_weight_train = compute_band_weights(y_train, X_train)

print("\nTraining samples by (label, band):")
print(pd.crosstab(y_train, X_train['band']))

tmp = pd.DataFrame({'label': y_train.values, 'band': X_train['band'].values, 'weight': sample_weight_train})
print("\nMean sample weight by (label, band):")
print(tmp.groupby(['label', 'band'], observed=True)['weight'].mean().unstack(fill_value=0).round(3))

___
## 2. Hyperparameter Search

### Grid Search with 5-fold Cross-Validation

Searches over a grid of XGBoost hyperparameters using `GridSearchCV` with 5-fold stratified cross-validation, scored by ROC-AUC.

$\rm \text{\textcolor{green}{What does 5-fold stratified cross-validation mean?}} $

ROC-AUC is used because it is invariant to class ratio — this data has a lens : non-lens ratio of ~1:50, compared to the expected survey ratio of 1 : 37,000.

You can try other scoring than ROC-AUC as well!

There are CV options other than grid search -- you should explore and implement these!; this is the simplest but takes the most amount of time.

You can also skip this step and manually set parameters in the next code block.

In [ ]:
### what kind of parameter space should you search? this would depnd on your data, the kind of problem
### you are solving. you should discuss with you neighbors, look up the GridSearchCV docs etc
param_grid = {
    'max_depth': ...,
    'learning_rate': ...,
    'subsample': ...,
    'colsample_bytree': ...
}

search = GridSearchCV(...)
search.fit(X_train, y_train, sample_weight=sample_weight_train)

best_params = search.best_params_
print(f"\nBest params: {best_params}")

___
## 3. Train & Evaluate Model

We train an XGBoost model using the best hyperparameters from the grid_search (or set `USE_BEST_PARAMS = False` if you'd like to set your own parameters manually). The manual parameters here are the best hyperparameters from a previous run.

We then valuate on a held-out test set. Note that the precision and recall are skewed by the large class imbalance.

In [ ]:
USE_BEST_PARAMS = False
### already know what the best parameters are? Fill them in here!
MANUAL_PARAMS = {
    'max_depth':        ...,
    'learning_rate':    ...,
    'subsample':        ...,
    'colsample_bytree': ...,
}

params = best_params if USE_BEST_PARAMS else MANUAL_PARAMS

model = XGBClassifier(
    n_estimators=..., # how many estimators should you use?
    val_metric='logloss',
    enable_categorical=..., # is there
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    **params,
)
model.fit(X_train, y_train, sample_weight=sample_weight_train,
          eval_set=[(X_val, y_val)], verbose=False)
print(f"Training complete. Best n_estimators: {model.best_iteration}")

HINT: If you don't know how a function works, you can simply call this:

In [ ]:
XGBClassifier?


Init signature:
XGBClassifier(
    *,
    objective: Union[str, xgboost.sklearn._SklObjWProto, Callable[[Any, Any], Tuple[numpy.ndarray, numpy.ndarray]], NoneType] = 'binary:logistic',
    **kwargs: Any,
) -> None
Docstring:     
Implementation of the scikit-learn API for XGBoost classification.
See :doc:`/python/sklearn_estimator` for more information.

Parameters
----------

    n_estimators : Optional[int]
        Number of boosting rounds.

    max_depth :  typing.Optional[int]

        Maximum tree depth for base learners.

    max_leaves : typing.Optional[int]

        Maximum number of leaves; 0 indicates no limit.

    max_bin : typing.Optional[int]

        If using histogram-based algorithm, maximum number of bins per feature

    grow_policy : typing.Optional[str]

        Tree growing policy.

        - depthwise: Favors splitting at nodes closest to the node,
        - lossguide: Favors splitting at nodes with highest loss change.

    learning_rate : typing.Optional[float

Now we evaluate test set metrics using the held-out test sets, X_test and y_test.

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-LAGN (0)', 'LAGN (1)'], digits=4))

### Visualization: Diagnostic Plots

Five diagnostic plots summarizing classifier performance on the held-out test set:

| Plot | Description |
|---|---|
| **Feature Importance** | XGBoost gain-based importance for each input feature |
| **ROC Curve** | True positive rate vs. false positive rate across all thresholds |
| **Precision-Recall Curve** | Precision vs. recall at the training-set class ratio (~1:50) |
| **P(LAGN) Distribution** | Predicted probability histograms for LAGN and non-LAGN DIASources |
| **Confusion Matrix** | Counts and row-normalized rates at nominal threshold (0.5) |

In [ ]:
# An example: Feature Importance
### here's how you would get the model feature importance
fig, ax = plt.subplots(figsize=(6, 5))
importance = model.feature_importances_
feat_names = np.array(X.columns.tolist())
top_idx = np.argsort(importance)[-20:]
ax.barh(feat_names[top_idx], importance[top_idx], color='steelblue', edgecolor='white')
ax.set_xlabel('Gain', fontsize=11)
ax.set_title('Feature Importance', fontsize=12, fontweight='bold')
ax.tick_params(axis='y', labelsize=8)
ax.spines[['top', 'right']].set_visible(False)

### keep in mind that you have imported some useful metric functions!
### you will need to use those to visualize your metrics
### this is a really important step of the problem -- this is what you will study and

### visualize the feature importance,
# the ROC curve,
# the precision-recall curve,
# the probability with which we classify LAGN vs non-LAGN
# the confusion matrix (make sure it is normalized)

In [ ]:
# Print feature importance
sorted_idx = np.argsort(importance)[::-1]
print('Full feature importance:')
for rank, i in enumerate(sorted_idx, 1):
    print(f"{rank:>3}. {feat_names[i]:<35} {importance[i]:.6f}")

In [ ]:
# SELECT FEATURES TO PLOT
SELECTED_FEATURES = [
    'snr',
    'x_y_err',
    'moment_ext',
    'ellip_ext',
    'flux_ext',
    'dipoleChi2',
    'dipoleLength',
    # Add or remove features as desired
]

FEATURE_PROPS = {
    'psf_fwhm':            ('linear', (0, 1),        'PSF FWHM'),
    'snr':                 ('linear', (0, 50),         'SNR'),
    'template_flux':       ('log',    (1e2, 1e7),      'Temp. Flux'),
    'scienceFlux':         ('log',    (1e2, 1e7),      'Sci. Flux'),
    'psfFlux':             ('log',    (1e2, 1e7),      'PSF Flux'),
    'apFlux':              ('log',    (1e2, 1e7),      'Ap. Flux'),
    'temp_sci_flux_ratio': ('linear', (0, 2),          'Temp./Sci.\nFlux'),
    'moment_ext':          ('linear', (0, 4),          'Moment Ext.'),
    'ellip_ext':           ('linear', (0, 1),          'Ellip. Diff.'),
    'flux_ext':            ('log',    (0.1, 10.0),     'Flux Ext.'),
    'extendedness':        ('linear', (0, 1),          'Rubin Ext.'),
    'psfChi2':             ('log',    (1, 1e5),        'PSF χ²'),
    'isDipole':            ('linear', (-0.1, 1.1),     'Is Dipole'),
    'dipoleFitAttempted':  ('linear', (-0.1, 1.1),     'Dip. Fit?'),
    'dipoleChi2':          ('log',    (1e2, 1e5),     'Dip. χ²'),
    'dipoleLength':        ('linear', (-0.1, 0.15),        'Dip. Length'),
    'x_y_err':             ('linear', (-1, 4),          'Cent. Err.'),
    'trailLength':         ('linear', (0, 4),          'Trail Length'),
    'trailFlux':           ('log',    (1e2, 1e7),      'Trail Flux'),
    'dipoleMeanFlux':      ('log',    (1e2, 1e7),      'Dipole Mean Flux'),
    'dipoleFluxDiffErr':   ('log',    (1e0, 1e5),      'Dipole Flux Diff Err'),
    'dipoleMeanFluxErr':   ('log',    (1e0, 1e5),      'Dipole Mean Flux Err'),
    'centroid_flag':       ('linear', (-0.1, 1.1),     'Centroid Flag'),
}

def to_array(subset, cols):
    arr = np.empty((len(subset), len(cols)), dtype=float)
    for i, c in enumerate(cols):
        scale, (lo, hi) = FEATURE_PROPS[c][0], FEATURE_PROPS[c][1]
        s = pd.to_numeric(subset[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
        if scale == 'log':
            s = s.where(s > 0)      # mask non-positive: log(≤0) is undefined
        med = s.median()
        fill = med if np.isfinite(med) else lo   # fall back to range lower bound
        s = s.fillna(fill)
        arr[:, i] = s.values
        # Final safety: clamp any surviving bad values
        if scale == 'log':
            bad = ~np.isfinite(arr[:, i]) | (arr[:, i] <= 0)
            arr[bad, i] = lo
        else:
            arr[~np.isfinite(arr[:, i]), i] = 0.0
    return arr

df_false = df[df['label'] == 0]
df_true  = df[df['label'] == 1]

XGB_THRESHOLD = 0.9691 #this is for 95% completeness, calculated from "Survey-Scale Analysis" below.
all_probs = model.predict_proba(X)[:, 1]
df_xgb = df[all_probs > XGB_THRESHOLD]
print(f"Sources passing XGBoost (threshold={XGB_THRESHOLD}): {len(df_xgb):,}")

# Filter candidate_cols to only include SELECTED_FEATURES
candidate_cols = [f for f in SELECTED_FEATURES if f in FEATURES and f != 'band']
print(f"Using {len(candidate_cols)} selected features: {candidate_cols}")

# Drop columns where any population has fewer than 2 unique in-range values
def is_plottable(col, *dfs, min_samples=2):
    scale, (lo, hi) = FEATURE_PROPS[col][0], FEATURE_PROPS[col][1]
    for d in dfs:
        s = pd.to_numeric(d[col], errors='coerce')
        if scale == 'log':
            s = s.where(s > 0)
        vals = s.dropna()
        in_range = vals[(vals >= lo) & (vals <= hi)]
        if in_range.nunique() < min_samples:
            return False
    return True

columns_corner = [c for c in candidate_cols if is_plottable(c, df_false, df_true, df_xgb)]
dropped = set(candidate_cols) - set(columns_corner)
if dropped:
    print(f"Dropped unplottable columns: {dropped}")

print(f"Final plotting columns ({len(columns_corner)}): {columns_corner}")

axes_scale = [FEATURE_PROPS[f][0] for f in columns_corner]
ranges     = [FEATURE_PROPS[f][1] for f in columns_corner]
labels     = [FEATURE_PROPS[f][2] for f in columns_corner]

data_array_all  = to_array(df_false, columns_corner)
data_array_lagn = to_array(df_true,  columns_corner)
data_array_xgb  = to_array(df_xgb,   columns_corner)

# Adjust figure size based on number of features
n_features = len(columns_corner)
fig_size = max(8, min(15, n_features * 2.5))  # Scale between 8 and 15

fig = corner.corner(data_array_all,
                    labels=labels,
                    axes_scale=axes_scale,
                    range=ranges,
                    fill_contours=True,
                    smooth=0.7,
                    show_titles=False,
                    color='grey',
                    plot_datapoints=False,
                    plot_contours=True,
                    plot_density=True,
                    bins=20,
                    fig=plt.figure(figsize=(fig_size, fig_size), dpi=300),
                    max_n_ticks=3,
                    hist_kwargs=dict(density=True)
                   )

fig = corner.corner(data_array_lagn,
                    labels=labels,
                    axes_scale=axes_scale,
                    range=ranges,
                    fill_contours=True,
                    smooth=0.7,
                    show_titles=False,
                    color='green',
                    plot_datapoints=False,
                    plot_contours=True,
                    plot_density=True,
                    bins=20,
                    fig=fig,
                    max_n_ticks=3,
                    hist_kwargs=dict(density=True)
                   )

fig = corner.corner(data_array_xgb,
                    labels=labels,
                    axes_scale=axes_scale,
                    range=ranges,
                    fill_contours=True,
                    smooth=0.7,
                    show_titles=False,
                    color='red',
                    plot_datapoints=False,
                    plot_contours=True,
                    plot_density=True,
                    bins=20,
                    fig=fig,
                    max_n_ticks=3,
                    hist_kwargs=dict(density=True)
                   )

for ax in fig.axes:
    ax.tick_params(labelsize=8, axis='both', which='major', pad=2)
    for label in ax.get_xticklabels():
        label.set_rotation(0)
    xlabels = [t for t in ax.xaxis.get_major_ticks() if t.label1.get_visible()]
    if xlabels:
        xlabels[-1].label1.set_visible(False)
    ylabels = [t for t in ax.yaxis.get_major_ticks() if t.label1.get_visible()]
    if ylabels:
        ylabels[0].label1.set_visible(False)

equation_text = (
    r'$\mathrm{Flux\ Ext.} = \dfrac{F_\mathrm{ap}}{F_\mathrm{PSF}}$' + '\n\n' +
    r'$\mathrm{Moment\ Ext.} = \dfrac{I_{xx} + I_{yy}}{I_{xx}^{\,\mathrm{PSF}} + I_{yy}^{\,\mathrm{PSF}}}$' + '\n\n' +
    r'$\mathrm{Ellip.\ Diff.} = \dfrac{\sqrt{(I_{xx}-I_{yy})^2+4I_{xy}^2}}{I_{xx}+I_{yy}} - '
    r'\dfrac{\sqrt{(I_{xx}^{\,\mathrm{PSF}}-I_{yy}^{\,\mathrm{PSF}})^2+4(I_{xy}^{\,\mathrm{PSF}})^2}}'
    r'{I_{xx}^{\,\mathrm{PSF}}+I_{yy}^{\,\mathrm{PSF}}}$'
)

# fig.text(0.55, 0.87, equation_text, fontsize=13,
#          bbox=dict(boxstyle='round,pad=0.8', facecolor='white', edgecolor='white', alpha=0),
#          verticalalignment='top')

legend_elements = [
    Patch(facecolor='grey',  edgecolor='black',    label=f'All DP1 DIASources ({len(df_false):,})'),
    Patch(facecolor='green', edgecolor='darkgreen', label=f'Injected LAGN ({len(df_true):,})'),
    Patch(facecolor='red',   edgecolor='darkred',   label=f'XGBoost p > {XGB_THRESHOLD} ({len(df_xgb):,})'),
]
fig.legend(handles=legend_elements, loc='upper right', fontsize=13, framealpha=0.9)

plt.show()

### Train & Save Final Model
We find the best n_estimators using early stopping, and then train a final model using all of the available data. This way, our model has seen as many lenses as possible when it gets applied to real data at ANTARES. Note that we did not do this during testing above, to avoid data leakage and make sure that we were testing on a set that the model had not seen before.

In [ ]:
# ── Train Final Model ──────────────────────────────────────────────────────────
USE_BEST_PARAMS = False

MANUAL_PARAMS = {
    'max_depth':        8,
    'learning_rate':    0.03,
    'subsample':        0.6,
    'colsample_bytree': 0.9,
}

params = best_params if USE_BEST_PARAMS else MANUAL_PARAMS

# Train with early stopping to find best n_estimators
print("Training with early stopping to find optimal n_estimators...")
model = XGBClassifier(
    n_estimators=500,
    eval_metric='logloss',
    enable_categorical=True,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    **params,
)
model.fit(X_train, y_train, sample_weight=sample_weight_train,
          eval_set=[(X_val, y_val)], verbose=False)
best_n_estimators = model.best_iteration
print(f"Best n_estimators: {best_n_estimators}")

# Retrain on train+val+test with the optimal n_estimators (no early stopping)
print(f"\nRetraining final model on train+val data with n_estimators={best_n_estimators}...")
X_train_final = pd.concat([X_train, X_val, X_test])
y_train_final = pd.concat([y_train, y_val, y_test])
sample_weight_val = compute_band_weights(y_val, X_val)
sample_weight_test = compute_band_weights(y_test, X_test)
sample_weight_final = np.concatenate([sample_weight_train, sample_weight_val, sample_weight_test])

final_model = XGBClassifier(
    n_estimators=best_n_estimators,  # Use the optimal value found
    eval_metric='logloss',
    enable_categorical=True,
    random_state=42,
    n_jobs=-1,
    **params,
)
final_model.fit(X_train_final, y_train_final, sample_weight=sample_weight_final, verbose=False)
print("Final model training complete.")

In [ ]:
# # Save the final model
# import pickle

# training_version = '2.0.7'
# model_filename = f'lantern_xgboost_t{training_version}.json'

# with open(model_filename, 'wb') as f:
#     pickle.dump(final_model, f)

# print(f"Model saved to {model_filename}")

---
## 4. Survey Scale Analysis

The test set metrics above reflect a ~1:50 class ratio, not the LSST DIASource LAGN to non-LAGN ratio of ~1:37,000. At survey scale, the same classifier will produce far more false positives relative to true positives.

Here we use only invariant quantities like true positive rate (completeness, TPR) and false positive rate (FPR) to prepare a representative completeness/purity curve.

## Survey Population Estimates

We estimate the number of LAGN and non-LAGN DIASources expected in the 10-year LSST survey and in year 1. These are rough calculations.

### LAGN DIASources

| Parameter | Value |
|---|---|
| Expected lensed AGN in full survey | 4,000 |
| Visits per sky position (over 10 years) | 800 |
| LAGN detection rate (DIASource per visit per LAGN) | 10% |
| **Total LAGN DIASources (10 yr)** | **320,000** |
| **Total LAGN DIASources (yr 1, 80 visits)** | **32,000** |

### Non-LAGN DIASources

Scaling Non-LAGN DIASources within a small field (ECDFS)  --> the LSST wide fast deep survey area.

| Parameter | Value |
|---|---|
| DP1 DIASources (ECDFS) | 551,975 |
| DP1 visits | 855 |
| DP1 field area | 0.785 deg² |
| DIASources / visit / deg² | 822.4 / visit / deg² |
| Full survey area | 18,000 deg² |
| **Total background DIASources (10 yr)** | **~11.8 billion** |
| **Total background DIASources (yr 1)** | **~1.18 billion** |


The ratio of LAGN to non-LAGN DIASources is **~1:37,000**.

### Completeness–Purity Analysis

We sweep the classification threshold and compute the completeness and purity at survey scale using 10-fold stratified cross-validation (for smoothness).

Completeness (TPR) and FPR are can be measured on the training data and then rescaled to yield counts of true positives and false positives:

$$N_\text{LAGN,yr1} = 32,000$$
$$N_\text{non-LAGN,yr1} = 1,184,257,459$$

$$\text{TP} = \text{TPR} \times N_\text{LAGN,yr1}$$
$$\text{FP} = \text{FPR} \times N_\text{non-LAGN,yr1}$$
$$\text{purity} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$

The purity is averaged across all 10 folds. We produce two sets of plots: one at the DIASource level, and one at the target level. In each case we plot:
1. **Completeness vs. Purity**
2. **LAGN DIASources vs. Purity** (how many LAGN DIASources detected at each purity level in year 1)

In [ ]:
lagn_diasources     = 32_000
non_lagn_diasources = 1_184_257_459
N_lenses_yr1 = 4_000

N_nonlenses_full = df[df['label'] == 0]['lens_id'].nunique()
N_nonlagn_diasources_full = (df['label'] == 0).sum()
avg_diasources_per_nonlens = N_nonlagn_diasources_full / N_nonlenses_full
N_nonlenses_yr1 = int(non_lagn_diasources / avg_diasources_per_nonlens)

print(f"\nLAGN DIASources     (yr 1) : {lagn_diasources:,}")
print(f"Non-LAGN DIASources (yr 1) : {non_lagn_diasources:,}")
print(f'For every lens we see, we will see {avg_diasources_per_nonlens}')


LAGN DIASources     (yr 1) : 32,000
Non-LAGN DIASources (yr 1) : 1,184,257,459


### Your goal now is to look at the completeness vs purity and sample numbers vs purity (i.e how many alerts do we have look through before we find a real lens!)

Example final plots are in this folder!

#### Completeness vs. Purity @ DIASource Level

#### Completeness vs. Purity @ Unique Object Level

### Once you've made it to the end of the notebook, record what you have learned! You should go back and experiment with different hyperparameters, including/excluding different features etc
You will present how well you model does in classifying lenses from non-lenses, just using alert stream data.
Talk about anything and everything you learned about!